### Chain of Thought (CoT)

En esta práctica usaremos la técnica de prompting llamada CoT (Chain of Thought) para mejorar las respuestas de un modelo de lenguaje grande (LLM) en tareas que requieren razonamiento complejo. CoT implica proporcionar al modelo ejemplos de razonamiento paso a paso en el prompt, lo que ayuda al modelo a descomponer problemas complejos en pasos más manejables.

In [40]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')

MODEL = "gemini-2.0-flash"
client = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta", api_key=google_api_key)

In [48]:
SYSTEM_PROMPT = """
    Usted es un asistente de IA experto en resolver consultas de usuarios utilizando el \"chain of thought\".
    Trabaja en los pasos START, PLAN y OUTPUT.
    Primero, debe PLANIFICAR lo que se debe hacer. El PLAN puede tener varios pasos.
    Una vez que considere que se ha PLANIFICADO lo suficiente, finalmente puede generar un OUTPUT.

    Reglas:
    - Siga estrictamente el formato de salida JSON proporcionado. No incluyas caracteres raros ni saltos de línea que puedan dar problema al parsear el JSON.
    - Ejecute solo un paso a la vez. La respuesta debe contener el siguiente paso ejecutado.
    - La secuencia de pasos es START (donde el usuario proporciona una entrada), PLAN (que puede ser varias veces) y finalmente OUTPUT (que se mostrará al usuario).
    - EL OUTPUT debe contener únicamente la respuesta sin explicaciones adicionales, ni cálculos, ni razonamiento.

    Formato de salida JSON:
    { \"step\": \"START\" | \"PLAN\" | \"OUTPUT\", \"content\": \"string\" }

    Ejemplo:
    START: Hola, ¿puede resolver 2 + 3 * 5 / 10?
    PLAN: { \"step\": \"PLAN\": \"content\": \"Parece que el usuario está interesado en un problema matemático\" }
    PLAN: { \"step\": \"PLAN\": \"content\": \"Al observar el problema, se debe resolver utilizando el método BODMAS\" }
    PLAN: { \"step\": \"PLAN\": \"content\": \"Sí, el método BODMAS es lo correcto aquí\" }
    PLAN: { \"step\": \"PLAN\": \"content\": \"Primero se debe multiplicar 3 * 5, lo que da 15\" }
    PLAN: { \"step\": \"PLAN\": \"content\": \"Ahora la nueva ecuación es 2 + 15 / 10\" }
    PLAN: { \"step\": \"PLAN\": \"content\": \"Se debe realizar la división, es decir, 15 / 10 = 1.5\" }
    PLAN: { \"step\": \"PLAN\": \"content\": \"Ahora la nueva ecuación es 2 + 1.5\" }
    PLAN: { \"step\": \"PLAN\": \"content\": \"Ahora, finalmente, se realiza la suma, que es 3.5\" }
    PLAN: { \"step\": \"PLAN\": \"content\": \"Excelente, se ha resuelto y finalmente se obtiene 3.5 como respuesta\" }
    OUTPUT: { \"step\": \"OUTPUT\": \"content\": \"3.5\" }
    
"""

In [49]:

message_history = [
    { "role": "system", "content": SYSTEM_PROMPT },
]


In [50]:
user_query = "👉🏻 Explica en español: (2+3*4)^2^3"
message_history.append({ "role": "user", "content": user_query })


In [51]:
print(user_query)
while True:
    response = client.chat.completions.create(
        model=MODEL,
        response_format={"type": "json_object"},
        messages=message_history
    )

    raw_result = response.choices[0].message.content
    #print("raw_result:", raw_result)
    message_history.append({"role": "assistant", "content": raw_result})
    
    parsed_result = json.loads(raw_result)

    if parsed_result.get("step") == "START":
        print("🔥", parsed_result.get("content"))
        continue

    if parsed_result.get("step") == "PLAN":
        print("🧠", parsed_result.get("content"))
        continue

    if parsed_result.get("step") == "OUTPUT":
        print("🤖", parsed_result.get("content"))
        break


👉🏻 Explica en español: (2+3*4)^2^3
🧠 Primero, necesito descomponer la expresión matemática para entender el orden de las operaciones.
🧠 El problema es (2+3*4)^2^3. Para resolverlo, seguiré el orden de las operaciones (PEMDAS/BODMAS).
🧠 Primero, resolveré el paréntesis: (2+3*4). Dentro del paréntesis, primero haré la multiplicación (3*4), y luego la suma.
🧠 Calcularé 3 * 4, lo que resulta en 12. Entonces el paréntesis se convierte en (2 + 12).
🧠 Ahora, calcularé 2 + 12, que es igual a 14. Así que la expresión se simplifica a 14^2^3.
🧠 Ahora, debo resolver la exponenciación. Tengo 14^2^3. La exponenciación se realiza de derecha a izquierda, así que primero calcularé 2^3.
🧠 Calcularé 2^3, lo que es igual a 8. Así que la expresión se simplifica a 14^8.
🧠 Finalmente, calcularé 14^8.
🤖 1475789056
